# Advanced Python Problems with Solutions: Properties and Descriptors

**One self-contained, executable notebook • 16 progressively harder problems • standard library only**

This notebook expands the supplied *Properties and Descriptors* lesson: `@property` versus `property(fget, fset)`, getter-only properties as **data descriptors**, attribute-lookup precedence, `__get__` / `__set__` / `__set_name__`, a custom `MakeProperty`, and decorator-style setter construction. Exercises advance to realistic validation, inheritance, `__slots__`, cached attributes, dynamic installation, observability, and a complete model.

**How to use:** Read each challenge, attempt it, then run the worked solution and assertions. Run **Kernel → Restart & Run All** for the full executable answer key. Python **3.10+**; no external runtime dependencies. Code cells contain tests rather than relying on visually inspected output. Independent examples use distinct class names to avoid accidental redefinition.

**Best-practice notes:** Treat `bool` carefully: `isinstance(True, int)` is true. Raise consistent, informative exceptions; validate *before* mutating state. Store per-instance values on the **instance**, never on a shared descriptor. Prefer built-in `property` unless reusable descriptor behavior is genuinely needed. Use timezone-aware `datetime.now(timezone.utc)` in new code rather than deprecated `datetime.utcnow()` patterns. The source's `MakeProperty.setter` mutates its original descriptor; Problem 8 adds property-style **copy-on-decoration** to avoid modifying a base class's descriptor.

## Notebook contract and lookup reference

For normal instance reads, a useful simplified ordering is:

1. Data descriptor found on the type / its MRO (`__get__` plus `__set__` **or** `__delete__`).
2. Instance `__dict__` entry, if an instance dictionary exists.
3. Non-data descriptor (`__get__` only), or other class attribute from the type / MRO.
4. `__getattr__`, if normal lookup fails.

Details: the order describes ordinary `object.__getattribute__`, not every possible custom override; `__slots__` can remove `__dict__`; **read-only** properties still have a `__set__` that rejects writes. An inherited descriptor follows the class MRO. Descriptor methods are generally invoked by attribute access, not by merely indexing an object's `__dict__`.

**Conventions for verification:** all `assert` statements must pass. Expected exceptions are checked explicitly; a missing exception fails the test. Do not use `assert` as a production validation mechanism: the examples' actual validation uses `raise`.

In [1]:
from contextlib import contextmanager
from datetime import datetime, timezone
from decimal import Decimal
from numbers import Integral
from weakref import WeakKeyDictionary
import inspect

@contextmanager
def raises(exception_type, message_fragment=None):
    """Tiny stdlib-only assertion helper for expected exceptions."""
    try:
        yield
    except exception_type as exc:
        if message_fragment is not None:
            assert message_fragment in str(exc), str(exc)
    else:
        raise AssertionError(f"Expected {exception_type.__name__}")

print("Setup complete: no third-party packages required.")

Setup complete: no third-party packages required.


---

## Problem 01 — A strict `age` property with safe state transitions

**Your challenge.** Implement `StrictPerson(age)` with a validated `@property`. Accept nonnegative integers, but reject `bool` and other numeric types, including `3.0`. A failed assignment must leave the previous valid age unchanged. Provide a readable docstring and prove `_age` is the actual storage key.

**Concepts:** validation, public API versus backing attribute, consistent errors.

Try an implementation before revealing/reading the solution below.

### Worked solution

Validate before writing. `bool` is a subclass of `int`, so rejecting it explicitly prevents `True` from silently becoming age 1. The setter keeps the public attribute governed by the data descriptor.

In [2]:
class StrictPerson:
    def __init__(self, age: int) -> None:
        self.age = age  # Route initialization through the setter.

    @property
    def age(self) -> int:
        """Age in completed years (a nonnegative, non-Boolean integer)."""
        return self._age

    @age.setter
    def age(self, value: int) -> None:
        if isinstance(value, bool) or not isinstance(value, Integral):
            raise TypeError("age must be a non-Boolean integer")
        if value < 0:
            raise ValueError("age must be nonnegative")
        self._age = int(value)

p01 = StrictPerson(27)
assert p01.age == 27
assert p01.__dict__ == {"_age": 27}
for invalid, error in [(-1, ValueError), (True, TypeError), (3.0, TypeError), ("10", TypeError)]:
    with raises(error):
        p01.age = invalid
    assert p01.age == 27  # Bad input does not corrupt old state.
p01.age = 0
assert p01.age == 0
assert "nonnegative" in StrictPerson.age.__doc__
print("Problem 01 passed")

Problem 01 passed


**Verification and edge cases.** Why not `self.age = value` inside the setter? That would recursively invoke the setter. `self._age` avoids recursion while preserving the public check.

---

## Problem 02 — Two construction syntaxes, one descriptor contract

**Your challenge.** Implement one range-checked temperature using `property(fget=..., fset=...)` and another using decorators. Both expose Celsius values at or above absolute zero (−273.15 °C). Demonstrate class-level access to the property object and verify that a getter-only property remains a data descriptor.

**Concepts:** property construction, descriptor introspection, read-only data descriptors.

Try an implementation before revealing/reading the solution below.

### Worked solution

Both syntaxes create built-in `property` objects. An optional setter changes the *behavior* of `__set__`, not whether the descriptor implements it.

In [3]:
ABSOLUTE_ZERO_C = -273.15

class ExplicitTemperature:
    def __init__(self, celsius: float):
        self.celsius = celsius

    def get_celsius(self):
        return self._celsius

    def set_celsius(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("celsius must be numeric, not bool")
        if value < ABSOLUTE_ZERO_C:
            raise ValueError("below absolute zero")
        self._celsius = float(value)

    celsius = property(fget=get_celsius, fset=set_celsius,
                       doc="Temperature in degrees Celsius")

class DecoratedTemperature:
    def __init__(self, celsius):
        self.celsius = celsius

    @property
    def celsius(self):
        """Temperature in degrees Celsius."""
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("celsius must be numeric, not bool")
        if value < ABSOLUTE_ZERO_C:
            raise ValueError("below absolute zero")
        self._celsius = float(value)

class ReadOnlyTemperature:
    @property
    def celsius(self):
        return 20.0

for cls in (ExplicitTemperature, DecoratedTemperature):
    t = cls(-273.15)
    assert t.celsius == ABSOLUTE_ZERO_C
    with raises(ValueError, "absolute zero"):
        t.celsius = -274
    assert t.celsius == ABSOLUTE_ZERO_C
    assert isinstance(cls.celsius, property)
assert hasattr(ReadOnlyTemperature.celsius, "__get__")
assert hasattr(ReadOnlyTemperature.celsius, "__set__")
with raises(AttributeError):
    ReadOnlyTemperature().celsius = 21
print("Problem 02 passed")

Problem 02 passed


**Verification and edge cases.** A built-in `property` is visible as a descriptor when accessed through the class, because its `__get__` returns itself for a class-level lookup.

---

## Problem 03 — Prove the lookup precedence rather than memorizing it

**Your challenge.** Create a data descriptor and a non-data descriptor, then plant same-named keys in an instance dictionary. Predict what reading each attribute returns. Prove that a normal class attribute can also be shadowed by the instance dictionary.

**Concepts:** data-descriptor precedence, non-data shadowing, MRO attribute lookup.

Try an implementation before revealing/reading the solution below.

### Worked solution

A descriptor with `__set__` is a data descriptor; the read is routed through it before the instance dictionary. A non-data descriptor yields to the instance dictionary.

In [4]:
class PriorityData:
    def __get__(self, instance, owner=None):
        return self if instance is None else "from data descriptor"

    def __set__(self, instance, value):
        instance.__dict__["_data_storage"] = value

class PriorityNonData:
    def __get__(self, instance, owner=None):
        return self if instance is None else "from non-data descriptor"

class PriorityDemo:
    data = PriorityData()
    nondata = PriorityNonData()
    plain = "from class"

sample03 = PriorityDemo()
assert sample03.data == "from data descriptor"
assert sample03.nondata == "from non-data descriptor"
assert sample03.plain == "from class"
sample03.__dict__.update(data="fake data", nondata="instance wins", plain="instance plain")
assert sample03.data == "from data descriptor"
assert sample03.nondata == "instance wins"
assert sample03.plain == "instance plain"
sample03.data = "new legitimate data"
assert sample03.__dict__["_data_storage"] == "new legitimate data"
assert sample03.__dict__["data"] == "fake data"
del sample03.__dict__["nondata"]
assert sample03.nondata == "from non-data descriptor"
print("Problem 03 passed")

Problem 03 passed


**Verification and edge cases.** The direct dictionary assignment is an *intentional experiment*, not an application pattern. It can put inconsistent data into an object without invoking any setter.

---

## Problem 04 — Read-only property, malicious shadowing, and deletion

**Your challenge.** Create a timezone-aware `observed_at` read-only property. Attempt an ordinary write, a direct dictionary shadow, and an ordinary deletion. Explain why only the raw dictionary can contain the forged string, while normal access still returns a datetime.

**Concepts:** read-only property and its rejecting `__set__`, descriptor priority.

Try an implementation before revealing/reading the solution below.

### Worked solution

A getter-only property implements a rejecting `__set__` and is therefore a data descriptor. For a normal built-in property without a deleter, deletion also raises `AttributeError`.

In [5]:
class Timestamped:
    @property
    def observed_at(self):
        return datetime.now(timezone.utc)

stamp04 = Timestamped()
with raises(AttributeError):
    stamp04.observed_at = "fake"
stamp04.__dict__["observed_at"] = "forged value"
assert stamp04.__dict__["observed_at"] == "forged value"
assert isinstance(stamp04.observed_at, datetime)
assert stamp04.observed_at.tzinfo is timezone.utc
with raises(AttributeError):
    del stamp04.observed_at
assert isinstance(Timestamped.observed_at, property)
print("Problem 04 passed")

Problem 04 passed


**Verification and edge cases.** This demonstrates lookup priority, **not** a security boundary. Code allowed to modify `__dict__` or the class can still mutate object internals.

---

## Problem 05 — Instrument the complete descriptor lifecycle

**Your challenge.** Implement a tracing data descriptor supporting `__set_name__`, `__get__`, `__set__`, and `__delete__`. Record owner/name at class creation, distinguish class access from instance access, and keep different instances isolated. Demonstrate that deleting one instance value does not affect the other.

**Concepts:** owner binding, storage per instance, read/write/delete protocol.

Try an implementation before revealing/reading the solution below.

### Worked solution

Store the resolved backing name on the descriptor, but store **values on each instance**. Returning `self` for class-level access preserves useful introspection.

In [6]:
class TraceField:
    def __set_name__(self, owner, name):
        self.public_name = name
        self.storage_name = f"_{name}"
        self.bound_owner = owner

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if self.storage_name not in instance.__dict__:
            raise AttributeError(f"{self.public_name} is unset")
        return instance.__dict__[self.storage_name]

    def __set__(self, instance, value):
        instance.__dict__[self.storage_name] = value

    def __delete__(self, instance):
        try:
            del instance.__dict__[self.storage_name]
        except KeyError:
            raise AttributeError(f"{self.public_name} is unset") from None

class Instrument:
    serial = TraceField()

inst_a, inst_b = Instrument(), Instrument()
field05 = Instrument.serial
assert field05.bound_owner is Instrument
assert field05.public_name == "serial"
assert isinstance(field05, TraceField)
with raises(AttributeError, "unset"):
    _ = inst_a.serial
inst_a.serial, inst_b.serial = "A-1", "B-2"
assert (inst_a.serial, inst_b.serial) == ("A-1", "B-2")
del inst_a.serial
with raises(AttributeError):
    _ = inst_a.serial
with raises(AttributeError):
    del inst_a.serial
assert inst_b.serial == "B-2"
print("Problem 05 passed")

Problem 05 passed


**Verification and edge cases.** A descriptor with only `__get__` and `__delete__` is *also* a data descriptor. `__set__` is sufficient but not strictly required for that classification.

---

## Problem 06 — A reusable validated descriptor used three times

**Your challenge.** Implement a `ValidatedField` that accepts a validator callable, uses `__set_name__` for the attribute name, raises `AttributeError` on missing values, and validates before storing. Build an `Employee` with three independently validated fields and prove multiple employees do not share data.

**Concepts:** descriptor reuse, missing-value semantics, instance isolation.

Try an implementation before revealing/reading the solution below.

### Worked solution

A single descriptor object belongs to its declaring *class*, but its values must be stored per *instance*. Make the validator return the normalized value; do not mutate state until it returns successfully.

In [7]:
class ValidatedField:
    def __init__(self, validator):
        self.validator = validator

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return instance.__dict__[self.storage_name]
        except KeyError:
            raise AttributeError(f"{self.name} is unset") from None

    def __set__(self, instance, value):
        normalized = self.validator(value)
        instance.__dict__[self.storage_name] = normalized

def nonempty_text(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError("expected nonempty text")
    return value.strip()

def valid_level(value):
    if type(value) is not int or not 1 <= value <= 10:
        raise ValueError("level must be an integer in [1, 10]")
    return value

def valid_email(value):
    if not isinstance(value, str) or value.count("@") != 1:
        raise ValueError("email must contain one @")
    return value.strip().lower()  # Deliberately minimal demo, not full email validation.

class Employee:
    name = ValidatedField(nonempty_text)
    level = ValidatedField(valid_level)
    email = ValidatedField(valid_email)

    def __init__(self, name, level, email):
        self.name, self.level, self.email = name, level, email

e1 = Employee("  Ada ", 4, "ADA@example.org")
e2 = Employee("Grace", 8, "grace@example.org")
assert (e1.name, e1.email) == ("Ada", "ada@example.org")
assert (e1.level, e2.level) == (4, 8)
with raises(ValueError):
    e1.level = True
assert e1.level == 4
assert e1.__dict__ == {"_name": "Ada", "_level": 4, "_email": "ada@example.org"}
assert Employee.name.name == "name" and Employee.level.name == "level"
print("Problem 06 passed")

Problem 06 passed


**Verification and edge cases.** This intentionally uses `__dict__` storage; Problem 11 covers slot-only instances. A production email validator would need a well-defined, domain-specific policy.

---

## Problem 07 — Diagnose the shared-state descriptor anti-pattern

**Your challenge.** Write a deliberately broken descriptor that stores the *latest value on itself*. Show how a second instance overwrites the first. Then repair the design without changing the consuming class’s public API.

**Concepts:** descriptor object lifetime, shared versus instance state, regression tests.

Try an implementation before revealing/reading the solution below.

### Worked solution

Descriptor instances are typically created once during class construction. Storing a user value in `self.value` therefore shares that value across all owner instances. Put it in `instance.__dict__` instead.

In [8]:
class BrokenShared:
    def __get__(self, instance, owner=None):
        return self if instance is None else getattr(self, "value", None)

    def __set__(self, instance, value):
        self.value = value  # BUG: belongs to descriptor, not owner instance.

class BrokenPair:
    item = BrokenShared()

broken_a, broken_b = BrokenPair(), BrokenPair()
broken_a.item = "first"
broken_b.item = "second"
assert broken_a.item == "second"  # Prove the bug, not desirable behavior.

class InstanceStored:
    def __set_name__(self, owner, name):
        self.storage = f"_{name}"

    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__.get(self.storage)

    def __set__(self, instance, value):
        instance.__dict__[self.storage] = value

class FixedPair:
    item = InstanceStored()

fixed_a, fixed_b = FixedPair(), FixedPair()
fixed_a.item, fixed_b.item = "first", "second"
assert (fixed_a.item, fixed_b.item) == ("first", "second")
assert FixedPair.item is not fixed_a.item
print("Problem 07 passed")

Problem 07 passed


**Verification and edge cases.** A single descriptor may hold immutable configuration (validator, name, documentation), but mutable per-object values belong in per-object storage.

---

## Problem 08 — Build a `property`-like descriptor with decorators and deletion

**Your challenge.** Upgrade the lesson’s `MakeProperty`: implement `__get__`, `__set__`, `__delete__`, `__set_name__`, `getter`, `setter`, and `deleter`; support a docstring; return **new descriptor instances** from decorator methods (like built-in `property`); use informative `AttributeError`s. Test read-only behavior, regular assignment, and deletion.

**Concepts:** descriptor construction, decorator chaining, copy-on-decoration, descriptor contracts.

Try an implementation before revealing/reading the solution below.

### Worked solution

`@MiniProperty` calls the constructor with the getter. `@x.setter` calls the descriptor’s method during class-body execution, replacing the class-local `x` with a cloned descriptor. `__set_name__` runs on the final descriptor after the class is created. Built-in `property` provides more behavior; this is an educational subset.

In [9]:
class MiniProperty:
    def __init__(self, fget=None, fset=None, fdel=None, doc=None):
        self.fget, self.fset, self.fdel = fget, fset, fdel
        self.__doc__ = doc if doc is not None else getattr(fget, "__doc__", None)
        self.name = getattr(fget, "__name__", "<unnamed>")

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        if self.fget is None:
            raise AttributeError(f"{self.name} is not readable")
        return self.fget(instance)

    def __set__(self, instance, value):
        if self.fset is None:
            raise AttributeError(f"{self.name} is not writable")
        self.fset(instance, value)

    def __delete__(self, instance):
        if self.fdel is None:
            raise AttributeError(f"{self.name} cannot be deleted")
        self.fdel(instance)

    def getter(self, func):
        return type(self)(func, self.fset, self.fdel, self.__doc__)

    def setter(self, func):
        return type(self)(self.fget, func, self.fdel, self.__doc__)

    def deleter(self, func):
        return type(self)(self.fget, self.fset, func, self.__doc__)

class SecretBox:
    @MiniProperty
    def secret(self):
        """Optional hidden value."""
        return getattr(self, "_secret", None)

    @secret.setter
    def secret(self, value):
        if not isinstance(value, str):
            raise TypeError("secret must be text")
        self._secret = value

    @secret.deleter
    def secret(self):
        if not hasattr(self, "_secret"):
            raise AttributeError("secret is unset")
        del self._secret

    @MiniProperty
    def identifier(self):
        return "fixed-ID"

box08 = SecretBox()
assert SecretBox.secret.fget.__name__ == "secret"
assert SecretBox.secret.__doc__ == "Optional hidden value."
box08.secret = "abc"
assert box08.secret == "abc"
with raises(TypeError):
    box08.secret = 123
assert box08.secret == "abc"
del box08.secret
assert box08.secret is None
with raises(AttributeError, "unset"):
    del box08.secret
with raises(AttributeError, "not writable"):
    box08.identifier = "change"
with raises(AttributeError, "cannot be deleted"):
    del box08.identifier
assert hasattr(SecretBox.identifier, "__set__")
print("Problem 08 passed")

Problem 08 passed


**Verification and edge cases.** Copy-on-decoration prevents mutating a descriptor another class still uses. The built-in implementation handles additional details; prefer `property` for everyday code.

---

## Problem 09 — Override only a property setter in a subclass

**Your challenge.** Define a base `Account` with a nonnegative integer `limit`. Make a `PremiumAccount` allow values up to 1,000,000 but keep base accounts capped at 10,000. Override the setter without mutating the base class’s property object. Confirm the getter remains inherited functionally.

**Concepts:** property cloning through inheritance, subclass customization.

Try an implementation before revealing/reading the solution below.

### Worked solution

`@BaseClass.property_name.setter` creates a *new* property on the subclass. The inherited `fget` remains available. Validation can differ without affecting the base class.

In [10]:
class BaseAccount:
    def __init__(self, limit):
        self.limit = limit

    @property
    def limit(self):
        """Account limit in whole units."""
        return self._limit

    @limit.setter
    def limit(self, value):
        if type(value) is not int or not 0 <= value <= 10_000:
            raise ValueError("base limit out of range")
        self._limit = value

class PremiumAccount(BaseAccount):
    @BaseAccount.limit.setter
    def limit(self, value):
        if type(value) is not int or not 0 <= value <= 1_000_000:
            raise ValueError("premium limit out of range")
        self._limit = value

base09 = BaseAccount(10_000)
premium09 = PremiumAccount(500_000)
assert (base09.limit, premium09.limit) == (10_000, 500_000)
with raises(ValueError, "base limit"):
    base09.limit = 500_000
assert base09.limit == 10_000
assert PremiumAccount.limit is not BaseAccount.limit
assert PremiumAccount.limit.fget is BaseAccount.limit.fget
assert PremiumAccount.limit.fset is not BaseAccount.limit.fset
assert PremiumAccount.limit.__doc__ == BaseAccount.limit.__doc__
print("Problem 09 passed")

Problem 09 passed


**Verification and edge cases.** Do not call the base setter from the premium setter: that would reimpose the base limit. A subclass choosing different invariants should document them clearly.

---

## Problem 10 — Keep a multi-field model valid with one atomic property setter

**Your challenge.** Implement a rectangle with positive finite numeric `width` and `height`, exposed through a single `dimensions` property. A tuple assignment must either update both fields or update neither; `area` should be computed and read-only. Test a malformed second value and an attempted area assignment.

**Concepts:** cross-field invariants, transactional property setters, computed properties.

Try an implementation before revealing/reading the solution below.

### Worked solution

Validate all incoming values into locals **before** committing any of them. Using a single setter also gives the caller a well-defined point for changing related fields.

In [11]:
import math

class Rectangle:
    def __init__(self, width, height):
        self.dimensions = (width, height)

    @staticmethod
    def _valid_dimension(value):
        if isinstance(value, bool) or not isinstance(value, (int, float)):
            raise TypeError("dimension must be a real number, not bool")
        if not math.isfinite(value) or value <= 0:
            raise ValueError("dimension must be positive and finite")
        return float(value)

    @property
    def dimensions(self):
        return self._width, self._height

    @dimensions.setter
    def dimensions(self, pair):
        try:
            width, height = pair
        except (TypeError, ValueError):
            raise ValueError("dimensions must contain exactly two values") from None
        width = self._valid_dimension(width)
        height = self._valid_dimension(height)
        self._width, self._height = width, height

    @property
    def area(self):
        return self._width * self._height

rect10 = Rectangle(2, 3)
assert rect10.area == 6.0
with raises(ValueError):
    rect10.dimensions = (10, -4)
assert rect10.dimensions == (2.0, 3.0)
with raises(ValueError):
    rect10.dimensions = (1, 2, 3)
with raises(ValueError):
    rect10.dimensions = (float("nan"), 2)
with raises(TypeError):
    rect10.dimensions = (True, 2)
with raises(AttributeError):
    rect10.area = 999
rect10.dimensions = [4, 5]
assert rect10.area == 20.0
print("Problem 10 passed")

Problem 10 passed


**Verification and edge cases.** The setter guarantees atomicity for its own validation failures in ordinary single-threaded use, **not** thread-safe atomicity under concurrent observers. Thread synchronization would be a separate concern.

---

## Problem 11 — Descriptors with `__slots__` and weak-key storage

**Your challenge.** Show why a descriptor that blindly accesses `instance.__dict__` fails with a slot-only class. Repair the design by (A) storing in a dedicated private slot and (B) using `WeakKeyDictionary` on weak-referenceable, hashable instances. Verify class access and multiple-instance isolation.

**Concepts:** slots, weak references, descriptor storage strategies, memory behavior.

Try an implementation before revealing/reading the solution below.

### Worked solution

A descriptor is not a storage allocator: its storage strategy must match the owner. A slot-based property can simply use `_value`. A weak-key map lets a general descriptor keep per-object data without `__dict__`, but owners must support weak references and hashing. Avoid storing a strong reference to the key *inside the value*, which would defeat weak-key cleanup.

In [12]:
class DictBacked:
    def __get__(self, instance, owner=None):
        return self if instance is None else instance.__dict__["_value"]

    def __set__(self, instance, value):
        instance.__dict__["_value"] = value

class SlottedFailure:
    __slots__ = ()
    value = DictBacked()

with raises(AttributeError):
    SlottedFailure().value = 1

class SlottedSuccess:
    __slots__ = ("_value",)

    def __init__(self, value):
        self.value = value

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, number):
        if type(number) is not int or number < 0:
            raise ValueError("value must be a nonnegative integer")
        self._value = number

slot_a, slot_b = SlottedSuccess(2), SlottedSuccess(5)
assert not hasattr(slot_a, "__dict__")
assert (slot_a.value, slot_b.value) == (2, 5)

class WeakStored:
    def __init__(self):
        self._values = WeakKeyDictionary()

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        try:
            return self._values[instance]
        except KeyError:
            raise AttributeError("value is unset") from None

    def __set__(self, instance, value):
        self._values[instance] = value

class WeakSlotted:
    __slots__ = ("__weakref__",)  # No __dict__, but weak-referenceable.
    value = WeakStored()

weak_a, weak_b = WeakSlotted(), WeakSlotted()
weak_a.value, weak_b.value = 11, 22
assert (weak_a.value, weak_b.value) == (11, 22)
assert not hasattr(weak_a, "__dict__")
assert isinstance(WeakSlotted.value._values, WeakKeyDictionary)
print("Problem 11 passed")

Problem 11 passed


**Verification and edge cases.** A `WeakKeyDictionary` is not universally applicable: a slot-only class needs `__weakref__`, and the instances must be hashable; dataclasses with equality but no hash may need a different design.

---

## Problem 12 — A cache that deliberately uses non-data descriptor precedence

**Your challenge.** Implement a `CachedOnce` descriptor exposing a computed value. On first read compute and store under the **public attribute name** in `instance.__dict__`; subsequent reads must find the cached value in the instance dictionary *before* invoking the descriptor. Allow cache invalidation by deleting that dictionary key. Contrast this with a read-only `property`.

**Concepts:** non-data descriptor precedence, memoization, invalidation.

Try an implementation before revealing/reading the solution below.

### Worked solution

Do **not** add `__set__` to `CachedOnce`: making it a data descriptor would prevent the instance dictionary from shadowing its `__get__`. This is a simplified educational cache, not a replacement for `functools.cached_property` (which has documented behavior and implementation details).

In [13]:
class CachedOnce:
    def __init__(self, function):
        self.function = function
        self.__doc__ = function.__doc__

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        value = self.function(instance)
        instance.__dict__[self.name] = value
        return value

class Metrics:
    def __init__(self):
        self.computations = 0

    @CachedOnce
    def expensive(self):
        """Compute once until invalidated."""
        self.computations += 1
        return 10 * self.computations

    @property
    def live(self):
        return 10 * self.computations

metrics12 = Metrics()
assert not hasattr(Metrics.expensive, "__set__")
assert metrics12.expensive == 10
assert metrics12.expensive == 10
assert metrics12.computations == 1
assert metrics12.__dict__["expensive"] == 10
metrics12.__dict__["live"] = -999
assert metrics12.live == 10  # Data descriptor overrides fake dict key.
del metrics12.__dict__["expensive"]
assert metrics12.expensive == 20
assert metrics12.computations == 2
print("Problem 12 passed")

Problem 12 passed


**Verification and edge cases.** This cache is not thread-safe and requires `__dict__`. It also permits `instance.expensive = other_value` because it is a non-data descriptor. Choose caching semantics deliberately.

---

## Problem 13 — Abstract properties and enforced interfaces

**Your challenge.** Declare an abstract `Shape.area` property. Prove the abstract base cannot be instantiated, implement it in a concrete circle, and verify that a class providing only an unrelated method remains abstract.

**Concepts:** `abc`, abstract properties, contracts versus validation.

Try an implementation before revealing/reading the solution below.

### Worked solution

Use `@property` outside `@abstractmethod` so the property object records abstractness through its getter. Abstractness is a class-construction/instantiation rule, not runtime return-type enforcement.

In [14]:
from abc import ABC, abstractmethod

class Shape(ABC):
    @property
    @abstractmethod
    def area(self):
        """Area in square units."""
        raise NotImplementedError

class Circle(Shape):
    def __init__(self, radius):
        if isinstance(radius, bool) or not isinstance(radius, (int, float)) or radius <= 0:
            raise ValueError("radius must be a positive number")
        self._radius = float(radius)

    @property
    def area(self):
        return math.pi * self._radius ** 2

class IncompleteShape(Shape):
    def perimeter(self):
        return 0

with raises(TypeError):
    Shape()
with raises(TypeError):
    IncompleteShape()
assert math.isclose(Circle(2).area, 4 * math.pi)
with raises(AttributeError):
    Circle(2).area = 0
assert "area" in Shape.__abstractmethods__
print("Problem 13 passed")

Problem 13 passed


**Verification and edge cases.** Implementing a concrete `area` property fulfills the abstract member; it does not automatically validate the numeric result. Validate that separately when required.

---

## Problem 14 — Dynamically attach a descriptor after class creation

**Your challenge.** Show that attaching a descriptor with `setattr(Class, name, descriptor)` after class creation does not automatically call `__set_name__`. Fix the installation explicitly and verify that the descriptor knows its public name and can serve multiple instances.

**Concepts:** class creation hooks, runtime descriptor registration.

Try an implementation before revealing/reading the solution below.

### Worked solution

Python invokes `__set_name__` automatically for descriptors in the namespace processed during `type.__new__`. Dynamically attached descriptors need a manual call (or a custom registration helper).

In [15]:
class LateField:
    def __set_name__(self, owner, name):
        self.name = name
        self.storage = f"_{name}"

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        return instance.__dict__[self.storage]

    def __set__(self, instance, value):
        instance.__dict__[self.storage] = value

class DynamicModel:
    pass

late14 = LateField()
setattr(DynamicModel, "identifier", late14)
assert not hasattr(late14, "name")  # No automatic name binding on setattr.
late14.__set_name__(DynamicModel, "identifier")
assert DynamicModel.identifier is late14
one14, two14 = DynamicModel(), DynamicModel()
one14.identifier, two14.identifier = 101, 202
assert (one14.identifier, two14.identifier) == (101, 202)
assert one14.__dict__ == {"_identifier": 101}
print("Problem 14 passed")

Problem 14 passed


**Verification and edge cases.** Calling `__set_name__` repeatedly with different names on the *same* descriptor can be unsafe if it stores only one name; use a fresh descriptor per distinct field or design for per-owner metadata.

---

## Problem 15 — Property access versus custom `__getattribute__`

**Your challenge.** Demonstrate that a class overriding `__getattribute__` and directly returning a value can bypass a property getter. Repair the override by delegating to `object.__getattribute__` and preserve the property’s validation/lookup semantics. Also explain why normal writes need the ordinary `__setattr__` path to call a property setter.

**Concepts:** attribute access customization, descriptor invocation, delegation.

Try an implementation before revealing/reading the solution below.

### Worked solution

Descriptor precedence belongs to the ordinary attribute-lookup implementation. A custom `__getattribute__` may bypass it by returning early. Delegate any unhandled name to the superclass; avoid recursive `self.name` calls inside the override.

In [16]:
class BypassesProperty:
    @property
    def value(self):
        return "computed by property"

    def __getattribute__(self, name):
        if name == "value":
            return "bypassed"  # Skips the usual descriptor lookup.
        return object.__getattribute__(self, name)

class DelegatesProperty:
    def __init__(self):
        self.value = 3

    @property
    def value(self):
        return self._value

    @value.setter
    def value(self, value):
        if type(value) is not int or value < 0:
            raise ValueError("value must be a nonnegative integer")
        self._value = value

    def __getattribute__(self, name):
        # Could instrument the read here; delegate to preserve descriptor behavior.
        return object.__getattribute__(self, name)

    def __setattr__(self, name, value):
        # Do not write to __dict__ directly here: that would bypass descriptors.
        object.__setattr__(self, name, value)

assert BypassesProperty().value == "bypassed"
good15 = DelegatesProperty()
assert good15.value == 3
with raises(ValueError):
    good15.value = -3
assert good15.value == 3
assert good15.__dict__ == {"_value": 3}
print("Problem 15 passed")

Problem 15 passed


**Verification and edge cases.** Ordinary `object.__setattr__` respects data descriptors. Overriding `__setattr__` is powerful; use it only when needed and delegate by default.

---

## Problem 16 — Capstone: transactional, reusable fields in a line-item model

**Your challenge.** Build a `Field` descriptor with validation and normalization, class access, delete support, and a clear missing-value error. Use it for `LineItem(name, quantity, unit_price, discount)`. Reject bool-as-integer, use exact `Decimal` values (reject float prices/rates), keep `total` computed and read-only, and implement `update(**changes)` that validates **every** requested change before committing **any**. Include at least ten assertions covering invalid inputs, independent instances, deletion, and atomic batch updates.

**Concepts:** realistic reusable descriptor API, precise money-like arithmetic, batch validation, invariants.

Try an implementation before revealing/reading the solution below.

### Worked solution

Have the field expose `normalize(value)` to support validating a batch without assigning early. Stage normalized private-key updates in a temporary dictionary, and perform one dictionary update only after all validation succeeds. Decimal avoids binary floating-point errors, although a real billing system must separately specify currencies, taxes, and rounding rules.

In [17]:
_MISSING = object()

class Field:
    """Educational per-instance validated descriptor for dict-backed models."""

    def __init__(self, validator):
        self.validator = validator
        self.name = None
        self.storage_name = None

    def __set_name__(self, owner, name):
        self.name = name
        self.storage_name = f"_{name}"

    def normalize(self, value):
        """Return a normalized value or raise; does not mutate an instance."""
        return self.validator(value)

    def __get__(self, instance, owner=None):
        if instance is None:
            return self
        result = instance.__dict__.get(self.storage_name, _MISSING)
        if result is _MISSING:
            raise AttributeError(f"{self.name} is not set")
        return result

    def __set__(self, instance, value):
        prepared = self.normalize(value)
        instance.__dict__[self.storage_name] = prepared

    def __delete__(self, instance):
        if self.storage_name not in instance.__dict__:
            raise AttributeError(f"{self.name} is not set")
        del instance.__dict__[self.storage_name]

def item_name(value):
    if not isinstance(value, str) or not value.strip():
        raise ValueError("name must be nonempty text")
    return value.strip()

def item_quantity(value):
    if type(value) is not int or value <= 0:
        raise ValueError("quantity must be a positive integer (not bool)")
    return value

def item_price(value):
    if not isinstance(value, Decimal) or not value.is_finite() or value < 0:
        raise ValueError("unit_price must be a finite nonnegative Decimal")
    return value

def item_discount(value):
    if not isinstance(value, Decimal) or not value.is_finite() or not Decimal(0) <= value <= Decimal(1):
        raise ValueError("discount must be a Decimal between 0 and 1")
    return value

class LineItem:
    name = Field(item_name)
    quantity = Field(item_quantity)
    unit_price = Field(item_price)
    discount = Field(item_discount)
    _field_names = frozenset({"name", "quantity", "unit_price", "discount"})

    def __init__(self, name, quantity, unit_price, discount=Decimal("0")):
        self.update(name=name, quantity=quantity,
                    unit_price=unit_price, discount=discount)

    def update(self, **changes):
        unknown = changes.keys() - self._field_names
        if unknown:
            raise TypeError(f"unknown fields: {', '.join(sorted(unknown))}")
        staged = {}
        for public_name, candidate in changes.items():
            descriptor = getattr(type(self), public_name)
            staged[descriptor.storage_name] = descriptor.normalize(candidate)
        self.__dict__.update(staged)  # Commit only after *all* validation.

    @property
    def total(self):
        return self.quantity * self.unit_price * (Decimal("1") - self.discount)

item16 = LineItem("  USB cable  ", 2, Decimal("19.99"), Decimal("0.10"))
other16 = LineItem("Mouse", 1, Decimal("15.00"))
assert item16.name == "USB cable"
assert item16.quantity == 2
assert item16.total == Decimal("35.9820")
assert other16.total == Decimal("15.00")
assert LineItem.quantity.name == "quantity"
assert isinstance(LineItem.total, property)
with raises(ValueError, "positive integer"):
    item16.quantity = True
with raises(ValueError, "Decimal"):
    item16.unit_price = 9.99
with raises(ValueError):
    item16.discount = Decimal("NaN")
with raises(ValueError):
    item16.name = "   "
assert (item16.name, item16.quantity, item16.unit_price) == (
    "USB cable", 2, Decimal("19.99"))
before16 = item16.__dict__.copy()
with raises(ValueError, "positive integer"):
    item16.update(name="Changed", unit_price=Decimal("12.00"), quantity=0)
assert item16.__dict__ == before16  # Whole batch rolls back by never committing.
with raises(TypeError, "unknown fields"):
    item16.update(color="red", name="Different")
assert item16.__dict__ == before16
with raises(AttributeError):
    item16.total = Decimal("0")
item16.update(quantity=3, discount=Decimal("0.25"))
assert item16.total == Decimal("44.9775")
assert other16.quantity == 1  # Independent instances.
assert LineItem.name is not LineItem.quantity

temporary16 = LineItem("Temporary", 1, Decimal("1"))
del temporary16.name
with raises(AttributeError, "not set"):
    _ = temporary16.name
with raises(AttributeError, "not set"):
    del temporary16.name
temporary16.name = "Restored"
assert temporary16.name == "Restored"
print("Problem 16 passed")

Problem 16 passed


**Verification and edge cases.** The raw `__dict__` remains intentionally accessible: descriptors enforce **normal attribute access**, not immutability against deliberate internal tampering. A production money model would define quantization and currency policy explicitly. The `update` guarantee applies to validation and commit within one thread, not concurrency or arbitrary side-effecting validators.

---

## Synthesis: what to remember

- Built-in properties are **data descriptors**, including read-only properties: a rejecting `__set__` still exists.
- A same-name instance dictionary entry shadows a **non-data descriptor**, not a data descriptor, under ordinary lookup.
- Use `@property` for individual fields, and custom descriptors when behavior is reused across many fields.
- `__set_name__` runs during class creation, not automatically on later `setattr` installation.
- Return the descriptor itself on class access (`instance is None`); keep mutable values per owner instance.
- Prefer copying property descriptors when changing accessor functions, particularly across inheritance.
- Choose storage deliberately for `__slots__`, weak references, and caching.
- Validate before mutation. For related fields, stage an entire batch and commit only after every check passes.

### Further independent extensions (with hints)

1. **Property-based tests:** generate invalid integers and decimals to challenge the capstone’s validators. *Hint:* keep core tests runnable without external dependencies; add Hypothesis only in a separate environment.
2. **Per-class descriptor reuse:** track `__set_name__` metadata for one descriptor shared between owner classes. *Hint:* a single `self.name` is insufficient; decide whether to prohibit reuse or map owner to metadata.
3. **Thread-safe caching:** protect concurrent cache filling with a lock, and define which lock applies to which instance. *Hint:* do not confuse ordinary single-thread behavior with synchronization.
4. **Financial precision:** define a currency and legal rounding mode before quantizing line totals. *Hint:* `Decimal('0.01')` and `quantize` are tools, not a complete policy.

**End of worked solutions.** Run all cells in order; the final checks are executable and signal a failure if any assertion is violated.